# schedule_raw_re_gifts
Incremental sync: `gifts` → `raw_re_gifts`

Replace `raw_re_gifts` with the UUID once available.

In [ ]:
%run user_configuration.ipynb
%run renxt_core.ipynb


In [ ]:
from typing import Iterable, Mapping, Callable

ENDPOINT_NAME = "gifts"
DATASET_ID    = "raw_re_gifts"   # replace with UUID if available
MERGE_KEY     = "id"

# Exact column list Domo expects — enforces order and presence before upsert
DEST_COLS = [
    'id', 'constituent_id', 'lookup_id', 'gift_status', 'type', 'subtype',
    'is_anonymous', 'date', 'date_added', 'date_modified', 'post_status',
    'post_date', 'reference', 'pulled_at_utc', 'acknowledgements',
    'gift_splits', 'payments', 'receipts', 'soft_credits', 'amount_value',
    'batch_number', 'constituency', 'fundraisers', 'linked_gifts',
    'balance_value', '_endpoint', '_batch_num',
]


In [ ]:
def domo_dtype_only_cast(
    df: pd.DataFrame,
    *,
    force_object_string_cols: Iterable[str] = ("id", "constituent_id", "lookup_id"),
    drop_timezone: bool = True,
    string_to_object: bool = True,
    boolean_to_object: bool = True,
    add_missing_columns: Iterable[str] = None,
    missing_defaults: Mapping[str, any] = None,
    target_dtypes: Mapping[str, str] = None,
) -> pd.DataFrame:
    """Prepare gifts DataFrame for Domo upsert:
    - Add any missing columns
    - Drop timezone from datetime columns
    - Convert StringDtype → object
    - Convert boolean → object
    - Force ID columns to object strings
    - Apply explicit target dtypes
    """
    out = df.copy()

    missing_defaults = dict(missing_defaults or {})
    target_dtypes    = dict(target_dtypes or {})

    if add_missing_columns:
        for col in add_missing_columns:
            if col not in out.columns:
                out[col] = missing_defaults.get(col, pd.NA)

    if drop_timezone:
        for col in out.columns:
            if pd.api.types.is_datetime64tz_dtype(out[col]):
                out[col] = out[col].dt.tz_convert("UTC").dt.tz_localize(None)

    if string_to_object:
        for col in out.columns:
            if pd.api.types.is_string_dtype(out[col]):
                out[col] = out[col].map(lambda x: x if pd.isna(x) else str(x)).astype("object")

    if boolean_to_object:
        for col in out.columns:
            if pd.api.types.is_bool_dtype(out[col]) or str(out[col].dtype) == "boolean":
                out[col] = out[col].astype("object")

    for col in force_object_string_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: x if pd.isna(x) else str(x)).astype("object")

    for col, dtype in target_dtypes.items():
        if col in out.columns:
            if dtype == "Int64":
                out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")
            elif dtype in ("float64", "Float64"):
                out[col] = pd.to_numeric(out[col], errors="coerce").astype("float64")
            else:
                out[col] = out[col].astype(dtype)

    return out


def ensure_columns(df: pd.DataFrame, columns: list, fill_value=pd.NA) -> pd.DataFrame:
    """Guarantee columns exist and enforce exact column order.
    Missing columns are added with fill_value. Extras are dropped.
    """
    df = df.copy()
    for c in columns:
        if c not in df.columns:
            df[c] = fill_value
    return df.reindex(columns=columns)


In [ ]:
# ── Step 1: fetch using watermark ─────────────────────────────────────────────
state_df = _read_pipeline_state()
since    = _get_since_date(ENDPOINT_NAME, state_df)

cfg     = ENDPOINT_LOOKUP[ENDPOINT_NAME]
cfg_run = dict(cfg)
cfg_run["params_base"] = dict(cfg.get("params_base") or {})
cfg_run["params_base"][cfg["incremental_candidates"][0]] = since
ENDPOINT_LOOKUP[ENDPOINT_NAME] = cfg_run

try:
    token_mgr = TokenManager(interactive=False)
    df_raw    = fetch_incremental(token_mgr=token_mgr, endpoint_name=ENDPOINT_NAME,
                                  days_back=0, session=requests.Session())
finally:
    ENDPOINT_LOOKUP[ENDPOINT_NAME] = cfg

if df_raw.empty:
    print("⚠️  No data returned — nothing to write.")
    _write_pipeline_state(ENDPOINT_NAME, "success", 0, state_df)
else:
    # ── Step 2: cast and conform to schema ────────────────────────────────────
    df_ready = domo_dtype_only_cast(
        df_raw,
        add_missing_columns=["balance_value", "_batch_num"],
        missing_defaults={"balance_value": pd.NA, "_batch_num": pd.NA},
        target_dtypes={"balance_value": "float64", "_batch_num": "Int64"},
    )
    df_publish = ensure_columns(df_ready, DEST_COLS)
    print(f"Rows to upsert: {len(df_publish):,}")
    print(df_publish.dtypes)

    # ── Step 3: upsert to Domo ────────────────────────────────────────────────
    try:
        domo.write_dataframe(
            df_publish,
            dataset=DATASET_ID,
            update_method="upsert",
            update_key=MERGE_KEY,
        )
        print(f"✅ Upserted {len(df_publish):,} rows → {DATASET_ID}")
        _write_pipeline_state(ENDPOINT_NAME, "success", len(df_publish), state_df)
    except Exception as e:
        print(f"❌ Upsert failed: {e}")
        _write_pipeline_state(ENDPOINT_NAME, "failed", 0, state_df)
        raise
